#隐喻识别

In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaConfig, RobertaModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import nltk
from nltk.corpus import wordnet as wn
import os
import logging
from torch.optim import AdamW
from types import SimpleNamespace
from collections import defaultdict
import pandas as pd
import numpy as np

# --- 准备工作 ---
logging.getLogger("datasets").setLevel(logging.INFO)
nltk.download('wordnet')
nltk.download('punkt')

# --- 模块1: 具体性评分与WordNet语义分析 ---
class ConcretenessScorer:
    def __init__(self, file_path):
        print(f"正在从 {file_path} 加载具体性评分...")
        try:
            df = pd.read_excel(file_path)
            word_col = 'Word'
            score_col = 'Conc.M'
            self.concreteness_dict = pd.Series(df[score_col].values, index=df[word_col].str.lower()).to_dict()
            print(f"成功加载 {len(self.concreteness_dict)} 个单词的具体性评分。")
        except FileNotFoundError:
            print(f"错误: 具体性评分文件未找到: {file_path}")
            raise
        except KeyError:
            print(f"错误: Excel文件中未找到 '{word_col}' 或 '{score_col}' 列。")
            raise

    def get_score(self, word):
        return self.concreteness_dict.get(str(word).lower(), 3.0)

    def get_wordnet_semantic_features(self, target_word, context_words):
        """
        计算目标词与上下文词的语义特征（基于WordNet上下位关系）。
        返回：路径距离、最低公共上位词深度、语义域差异。
        """
        def get_hypernym_path(synset):
            """获取synset的hypernym路径直到根节点"""
            path = []
            current = synset
            while current:
                path.append(current)
                hypernyms = current.hypernyms()
                current = hypernyms[0] if hypernyms else None
            return path

        features = []
        target_synsets = wn.synsets(target_word)
        if not target_synsets:
            return [0.0, 0.0, 0.0]  # 默认值

        target_synset = target_synsets[0]  # 取第一个义项
        target_path = get_hypernym_path(target_synset)

        path_distances = []
        lch_depths = []
        for context_word in context_words:
            context_synsets = wn.synsets(context_word)
            if not context_synsets:
                continue
            context_synset = context_synsets[0]
            context_path = get_hypernym_path(context_synset)

            # 计算路径距离
            try:
                path_distance = target_synset.shortest_path_distance(context_synset)
                path_distance = path_distance if path_distance is not None else 10.0
            except:
                path_distance = 10.0
            path_distances.append(path_distance)

            # 计算最低公共上位词（LCH）深度
            try:
                lch = target_synset.lowest_common_hypernyms(context_synset)
                lch_depth = lch[0].max_depth() if lch else 0.0
            except:
                lch_depth = 0.0
            lch_depths.append(lch_depth)

        # 聚合特征
        avg_path_distance = np.mean(path_distances) if path_distances else 0.0
        avg_lch_depth = np.mean(lch_depths) if lch_depths else 0.0
        semantic_domain_diff = 1.0 if avg_path_distance > 5.0 else 0.0  # 阈值判断跨域
        return [float(avg_path_distance), float(avg_lch_depth), float(semantic_domain_diff)]

# --- 模块2: 数据集加载与处理 ---
def load_vua20_data(cache_dir="./dataset_cache"):
    try:
        ds = load_dataset("CreativeLang/vua20_metaphor", cache_dir=cache_dir)
        print("VUA20 数据集加载成功:", ds)
        return ds
    except Exception as e:
        print(f"加载数据集失败: {e}")
        raise

pos_to_id = defaultdict(lambda: len(pos_to_id))
id_to_pos = {}
common_pos_tags = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "ADP", "NUM", "CONJ", "INTJ", "PART", "SCONJ", "SYM", "X", ".", "AUX", "PROPN", "PUNCT", "SPACE", "CCONJ"]
for tag in common_pos_tags:
    pos_to_id[tag]

class MetaphorDataset(Dataset):
    def __init__(self, data, tokenizer, concreteness_scorer, max_len=128):
        self.sentences = [item['sentence'] for item in data]
        self.w_indices = [item['w_index'] for item in data]
        self.labels = [item['label'] for item in data]
        self.pos_tags_str = [item['POS'] for item in data]
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.original_sentences = [item['sentence'] for item in data]
        self.concreteness_scorer = concreteness_scorer
        if self.concreteness_scorer is None:
            raise ValueError("ConcretenessScorer 实例必须提供。")
        self.pos_ids = [pos_to_id[tag] for tag in self.pos_tags_str]
        for tag, idx in pos_to_id.items():
            id_to_pos[idx] = tag

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        w_index = self.w_indices[idx]
        label = self.labels[idx]
        pos_id = self.pos_ids[idx]
        original_sentence = self.original_sentences[idx]

        words = sentence.split()
        target_word = ""
        literal_meaning = ""
        if 0 <= w_index < len(words):
            target_word = words[w_index]
            synsets = wn.synsets(target_word)
            literal_meaning = synsets[0].definition() if synsets else ""
            words[w_index] = f"[TARGET]{target_word}[/TARGET]"
            modified_sentence = " ".join(words)
            modified_sentence += f" [LITERAL] {literal_meaning}"
        else:
            modified_sentence = sentence

        encoding = self.tokenizer(modified_sentence, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        input_ids = encoding['input_ids'].flatten()

        target_mask = torch.zeros(self.max_len, dtype=torch.float)
        target_tokens_str = self.tokenizer.encode(f"[TARGET]{target_word}[/TARGET]", add_special_tokens=False)
        for i in range(len(input_ids) - len(target_tokens_str) + 1):
            if torch.equal(input_ids[i:i+len(target_tokens_str)], torch.tensor(target_tokens_str)):
                target_mask[i:i+len(target_tokens_str)] = 1.0
                break

        target_sentence = target_word if target_word else "[PAD]"
        encoding_2 = self.tokenizer(target_sentence, add_special_tokens=True, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        input_ids_2 = encoding_2['input_ids'].flatten()

        target_mask_2 = torch.zeros(self.max_len, dtype=torch.float)
        target_tokens_2 = self.tokenizer.encode(target_word, add_special_tokens=False)
        for i in range(len(input_ids_2) - len(target_tokens_2) + 1):
            if torch.equal(input_ids_2[i:i+len(target_tokens_2)], torch.tensor(target_tokens_2)):
                target_mask_2[i:i+len(target_tokens_2)] = 1.0
                break

        # 计算具体性特征
        target_concreteness_score = self.concreteness_scorer.get_score(target_word)
        context_words = [word for i, word in enumerate(original_sentence.split()) if i != w_index]
        if context_words:
            context_concreteness_score = sum(self.concreteness_scorer.get_score(w) for w in context_words) / len(context_words)
        else:
            context_concreteness_score = 3.0

        # 计算WordNet语义特征
        semantic_features = self.concreteness_scorer.get_wordnet_semantic_features(target_word, context_words)
        concreteness_features = torch.tensor([
            target_concreteness_score,
            context_concreteness_score,
            abs(target_concreteness_score - context_concreteness_score)
        ] + semantic_features, dtype=torch.float)

        return {
            'input_ids': input_ids,
            'input_ids_2': input_ids_2,
            'target_mask': target_mask,
            'target_mask_2': target_mask_2,
            'attention_mask': encoding['attention_mask'].flatten(),
            'attention_mask_2': encoding_2['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'pos_ids': torch.tensor(pos_id, dtype=torch.long),
            'original_sentence': original_sentence,
            'concreteness_features': concreteness_features
        }

# --- 新增: 监督对比学习损失函数 (SupCon) with Hard Negatives ---
def supcon_loss(features, labels, temperature=0.07, hard_neg_ratio=0.5, focal_gamma=1.5):
    """
    计算监督对比损失 (SupCon) 并加入硬负样本。
    - features: (batch_size, hidden_dim)
    - labels: (batch_size,)
    - hard_neg_ratio: 硬负样本占负样本比例
    """
    device = features.device
    features = nn.functional.normalize(features, dim=1)
    batch_size = features.shape[0]
    labels = labels.contiguous().view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)  # 正样本掩码

    # 相似度矩阵
    anchor_dot_contrast = torch.div(torch.matmul(features, features.T), temperature)

    # 数值稳定性
    logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
    logits = anchor_dot_contrast - logits_max.detach()

    # 掩码排除自身
    logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
    mask = mask * logits_mask

    # 选择硬负样本
    neg_mask = (1.0 - mask) * logits_mask  # 所有负样本掩码
    neg_sim = anchor_dot_contrast * neg_mask  # 负样本相似度
    _, hard_neg_indices = torch.topk(neg_sim, k=int(batch_size * hard_neg_ratio), dim=1, largest=True)  # 选择最相似的硬负样本
    hard_neg_mask = torch.zeros_like(neg_mask).scatter_(1, hard_neg_indices, 1.0)

    # 结合硬负样本和正样本计算损失
    final_mask = mask + (neg_mask * hard_neg_mask)  # 正样本 + 硬负样本
    exp_logits = torch.exp(logits) * final_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    # 正样本的平均 log-likelihood
    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-9)
    # 硬负样本的贡献（可选加权）
    mean_log_prob_neg = (hard_neg_mask * log_prob).sum(1) / (hard_neg_mask.sum(1) + 1e-9)
    prob_pos = torch.exp(mean_log_prob_pos)
    focal_weight = (1 - prob_pos) ** focal_gamma
    loss = - (focal_weight * mean_log_prob_pos).mean() - 0.1 * mean_log_prob_neg.mean()

    # 总损失
    # loss = -mean_log_prob_pos.mean() - 0.1 * mean_log_prob_neg.mean()  # 加权硬负样本损失
    return loss

# --- 模块3: 自定义RoBERTa模型 ---
class MetaphorRoberta(nn.Module):
    def __init__(self, model_name='roberta-large', num_labels=2, num_pos_tags=len(pos_to_id), pos_embedding_dim=64, num_concreteness_features=6,
                 contrastive_weight=0.1, temperature=0.5):
        super(MetaphorRoberta, self).__init__()
        self.num_labels = num_labels
        self.config = RobertaConfig.from_pretrained(model_name)
        self.encoder = RobertaModel.from_pretrained(model_name, config=self.config)
        self.args = SimpleNamespace(drop_ratio=0.1, classifier_hidden=128, small_mean=True)
        self.dropout = nn.Dropout(self.args.drop_ratio)
        self.pos_embedding = nn.Embedding(num_embeddings=num_pos_tags, embedding_dim=pos_embedding_dim)
        self.self_attention = nn.MultiheadAttention(embed_dim=self.config.hidden_size, num_heads=8, dropout=self.args.drop_ratio, batch_first=True)

        input_dim_for_target_with_pos = self.config.hidden_size + pos_embedding_dim
        self.SPV_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)
        self.MIP_linear = nn.Linear(self.config.hidden_size + input_dim_for_target_with_pos, self.args.classifier_hidden)

        gate_input_dim = self.config.hidden_size + input_dim_for_target_with_pos
        self.gate_mlp = nn.Sequential(
            nn.Linear(gate_input_dim, self.args.classifier_hidden),
            nn.ReLU(),
            nn.Dropout(self.args.drop_ratio),
            nn.Linear(self.args.classifier_hidden, 2)
        )

        linguistic_feature_dim = 16
        self.linguistic_mlp = nn.Sequential(
            nn.Linear(num_concreteness_features, linguistic_feature_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(linguistic_feature_dim * 2, linguistic_feature_dim)
        )

        self.classifier = nn.Linear(self.args.classifier_hidden + linguistic_feature_dim, num_labels)
        self.logsoftmax = nn.LogSoftmax(dim=1)

        self.contrastive_weight = contrastive_weight
        self.temperature = temperature

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.pos_embedding.weight, mean=0.0, std=self.config.initializer_range)
        for module in [self.SPV_linear, self.MIP_linear, self.classifier]:
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        for mlp in [self.gate_mlp, self.linguistic_mlp]:
            for module in mlp:
                if isinstance(module, nn.Linear):
                    module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
                    if module.bias is not None:
                        module.bias.data.zero_()

    def forward(self, input_ids, input_ids_2, target_mask, target_mask_2, attention_mask, attention_mask_2, labels=None, pos_ids=None, concreteness_features=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        attn_output, _ = self.self_attention(sequence_output, sequence_output, sequence_output, key_padding_mask=~attention_mask.bool())
        sequence_output = attn_output
        cls_output = sequence_output[:, 0, :]

        target_output = sequence_output * target_mask.unsqueeze(2)
        target_output = self.dropout(target_output)
        target_output = target_output.sum(dim=1) / (target_mask.sum(-1, keepdim=True) + 1e-10)

        pos_embedding = self.pos_embedding(pos_ids)
        target_output_with_pos = torch.cat([target_output, pos_embedding], dim=1)

        outputs_2 = self.encoder(input_ids=input_ids_2, attention_mask=attention_mask_2)
        sequence_output_2 = outputs_2.last_hidden_state
        target_output_2 = sequence_output_2 * target_mask_2.unsqueeze(2)
        target_output_2 = self.dropout(target_output_2)
        target_output_2 = target_output_2.sum(dim=1) / (target_mask_2.sum(-1, keepdim=True) + 1e-10)

        SPV_hidden = self.SPV_linear(torch.cat([cls_output, target_output_with_pos], dim=1))
        MIP_hidden = self.MIP_linear(torch.cat([target_output_2, target_output_with_pos], dim=1))

        gate_input = torch.cat([cls_output, target_output_with_pos], dim=1)
        gate_weights = torch.sigmoid(self.gate_mlp(gate_input))
        SPV_weight, MIP_weight = gate_weights[:, 0:1], gate_weights[:, 1:2]
        combined_hidden = SPV_weight * SPV_hidden + MIP_weight * MIP_hidden

        if concreteness_features is None:
            raise ValueError("模型 forward 函数需要 concreteness_features。")

        linguistic_embedding = self.linguistic_mlp(concreteness_features)
        final_features = torch.cat([combined_hidden, linguistic_embedding], dim=1)

        contrastive_loss_value = torch.tensor(0.0, device=final_features.device)
        if self.training and labels is not None:
            contrastive_loss_value = supcon_loss(final_features, labels, temperature=self.temperature)

        logits = self.classifier(self.dropout(final_features))
        logits = self.logsoftmax(logits)

        if labels is not None:
            loss_fct = nn.NLLLoss()
            main_loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            total_loss = main_loss + self.contrastive_weight * contrastive_loss_value
            return {'loss': total_loss, 'logits': logits, 'main_loss': main_loss, 'contrastive_loss': contrastive_loss_value}
        return {'logits': logits}

# --- 模块4: 构建与训练流程 ---
def build_model(model_name='roberta-large', num_labels=2):
    tokenizer = RobertaTokenizer.from_pretrained(model_name)
    tokenizer.add_tokens(['[TARGET]', '[/TARGET]', '[LITERAL]', '[POS]'])
    model = MetaphorRoberta(model_name, num_labels, num_pos_tags=len(pos_to_id))
    model.encoder.resize_token_embeddings(len(tokenizer))
    return model, tokenizer

def train_model(model, train_loader, val_loader, device, tokenizer, epochs=4):
    optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
    model.to(device)

    best_f1 = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_main_loss = 0
        total_contrastive_loss = 0
        train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for batch in train_progress:
            input_ids = batch['input_ids'].to(device)
            input_ids_2 = batch['input_ids_2'].to(device)
            target_mask = batch['target_mask'].to(device)
            target_mask_2 = batch['target_mask_2'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            attention_mask_2 = batch['attention_mask_2'].to(device)
            labels = batch['labels'].to(device)
            pos_ids = batch['pos_ids'].to(device)
            concreteness_features = batch['concreteness_features'].to(device)

            outputs = model(
                input_ids=input_ids, input_ids_2=input_ids_2,
                target_mask=target_mask, target_mask_2=target_mask_2,
                attention_mask=attention_mask, attention_mask_2=attention_mask_2,
                labels=labels, pos_ids=pos_ids,
                concreteness_features=concreteness_features
            )
            loss = outputs['loss']
            main_loss = outputs['main_loss']
            contrastive_loss = outputs['contrastive_loss']
            total_loss += loss.item()
            total_main_loss += main_loss.item()
            total_contrastive_loss += contrastive_loss.item()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            train_progress.set_postfix({'total_loss': loss.item(), 'main_loss': main_loss.item(), 'ctr_loss': contrastive_loss.item()})

        avg_loss = total_loss / len(train_loader)
        avg_main_loss = total_main_loss / len(train_loader)
        avg_contrastive_loss = total_contrastive_loss / len(train_loader)
        print(f"\nEpoch {epoch+1}/{epochs}, Average Total Loss: {avg_loss:.4f}, Avg Main Loss: {avg_main_loss:.4f}, Avg Contrastive Loss: {avg_contrastive_loss:.4f}")

        model.eval()
        val_preds, val_labels = [], []
        val_progress = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validate]", leave=False)
        with torch.no_grad():
            for batch in val_progress:
                input_ids = batch['input_ids'].to(device)
                input_ids_2 = batch['input_ids_2'].to(device)
                target_mask = batch['target_mask'].to(device)
                target_mask_2 = batch['target_mask_2'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                attention_mask_2 = batch['attention_mask_2'].to(device)
                labels = batch['labels'].to(device)
                pos_ids = batch['pos_ids'].to(device)
                concreteness_features = batch['concreteness_features'].to(device)

                outputs = model(
                    input_ids=input_ids, input_ids_2=input_ids_2,
                    target_mask=target_mask, target_mask_2=target_mask_2,
                    attention_mask=attention_mask, attention_mask_2=attention_mask_2,
                    pos_ids=pos_ids,
                    concreteness_features=concreteness_features
                )
                preds = torch.argmax(outputs['logits'], dim=1)
                val_preds.extend(preds.cpu().tolist())
                val_labels.extend(labels.cpu().tolist())

        val_acc = accuracy_score(val_labels, val_preds)
        val_precision = precision_score(val_labels, val_preds, pos_label=1, zero_division=0)
        val_recall = recall_score(val_labels, val_preds, pos_label=1, zero_division=0)
        val_f1 = f1_score(val_labels, val_preds, pos_label=1, zero_division=0)
        print(f"Validation Metrics - Accuracy: {val_acc:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            model_save_dir = "/content/drive/MyDrive/PhD/Metaphor_model/Metaphor_model"
            os.makedirs(model_save_dir, exist_ok=True)
            model.encoder.save_pretrained(model_save_dir)
            tokenizer.save_pretrained(model_save_dir)
            torch.save(model.state_dict(), os.path.join(model_save_dir, "pytorch_model.bin"))
            print(f"New best model saved to {model_save_dir} with F1: {best_f1:.4f}")

# --- 模块5: 主函数 ---
def main():
    cache_dir = "./dataset_cache"
    os.makedirs(cache_dir, exist_ok=True)
    path_to_concreteness_data = "/content/drive/MyDrive/PhD/Metaphor_model/Concreteness ratings.xlsx"
    concreteness_scorer = ConcretenessScorer(path_to_concreteness_data)
    dataset = load_vua20_data(cache_dir=cache_dir)
    train_data = dataset['train']
    val_data = dataset['test']
    model, tokenizer = build_model()
    train_dataset = MetaphorDataset(train_data, tokenizer, concreteness_scorer=concreteness_scorer)
    val_dataset = MetaphorDataset(val_data, tokenizer, concreteness_scorer=concreteness_scorer)
    train_labels = [item['label'] for item in train_data]
    class_counts = torch.bincount(torch.tensor(train_labels))
    weights = 1.0 / class_counts.float()
    sample_weights = [weights[label] for label in train_labels]
    sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))
    train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=32)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"模型将在 {'GPU: ' + torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'} 上运行")
    train_model(model, train_loader, val_loader, device, tokenizer, epochs=4)

if __name__ == "__main__":
    main()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


正在从 /content/drive/MyDrive/PhD/Metaphor_model/Concreteness ratings.xlsx 加载具体性评分...
成功加载 39954 个单词的具体性评分。


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Generating dataset vua20_metaphor (/content/dataset_cache/CreativeLang___vua20_metaphor/default/0.0.0/47c04abe372718066681b600f65db85cf8a4ff4b)
INFO:datasets.builder:Generating dataset vua20_metaphor (/content/dataset_cache/CreativeLang___vua20_metaphor/default/0.0.0/47c04abe372718066681b600f65db85cf8a4ff4b)
INFO:datasets.builder:Downloading and preparing dataset vua20_metaphor/default (download: 31.03 MiB, generated: 35.12 MiB, post-processed: Unknown size, total: 66.16 MiB) to /content/dataset_cache/CreativeLang___vua20_metaphor/default/0.0.0/47c04abe372718066681b600f65db85cf8a4ff4b...


...

Validation Metrics - Accuracy: 0.8976, Precision: 0.7017, Recall: 0.7471, F1: 0.7237
New best model saved to /content/drive/MyDrive/PhD/Metaphor_model/Metaphor_model_ConcreteMLP4 with F1: 0.7237



Epoch 3/4, Average Total Loss: 0.3493, Avg Main Loss: 0.0659, Avg Contrastive Loss: 2.8342


Validation Metrics - Accuracy: 0.9096, Precision: 0.7672, Recall: 0.7125, F1: 0.7388
New best model saved to /content/drive/MyDrive/PhD/Metaphor_model/Metaphor_model_ConcreteMLP4 with F1: 0.7388



Epoch 4/4, Average Total Loss: 0.3152, Avg Main Loss: 0.0358, Avg Contrastive Loss: 2.7938


Validation Metrics - Accuracy: 0.9109, Precision: 0.7904, Recall: 0.6848, F1: 0.7339
